In [ ]:
import pybamm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import math
import dfols
import signal
from scipy.integrate import solve_ivp
from scipy.fft import fft, fftfreq, fftshift
from scipy.signal import savgol_filter
from scipy.signal import find_peaks
from scipy import interpolate, integrate
from stopit import threading_timeoutable as timeoutable
import os, sys
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))
from batfuns import *
plt.rcParams = set_rc_params(plt.rcParams)
import winsound
from pybamm import exp, constants, Parameter
import pickle
from tqdm import tqdm

eSOH_DIR = "../data/esoh_R/"
oCV_DIR = "../data/ocv/"
cyc_DIR = "../data/cycling/"
fig_DIR = "../figures/figures_model/"
res_DIR = "../data/results_paper/"
resistance_DIR = "../data/resistance/"
%matplotlib widget

In [ ]:
def plot_fun(data,n,cell_no):
    df = data[data["Cycle number"]==n]
    df= df.reset_index(drop=True)
    fig,ax = plt.subplots(2,2,figsize=(8,6))
    ax1 = ax.flat[0]
    ax1.plot(df["Time [s]"],df["Voltage [V]"])
    ax1.set_ylabel("Voltage [V]")
    ax1.set_xlabel("Time [s]")
    ax2 = ax.flat[2]
    ax2.plot(df["Time [s]"],df["Current [mA]"])
    ax2.set_ylabel("Current [mA]")
    ax2.set_xlabel("Time [s]")
    ax3 = ax.flat[1]
    ax3.plot(df["Time [s]"],df["Expansion [mu m]"])
    ax3.set_ylabel(r"Expansion [$\mu$m]")
    ax3.set_xlabel("Time [s]")
    ax4 = ax.flat[3]
    ax4.plot(df["Time [s]"],df["Temperature [C]"])
    ax4.set_ylabel(r"Temperature [$^\circ$C]")
    ax4.set_xlabel("Time [s]")
    fig.suptitle(f"Cell: {cell_no}, Cycle #{n}")
    fig.tight_layout()
    return fig

In [ ]:
for cell in range(26,36):
    cell_no = f'{cell:02d}'
    data = pd.read_csv(cyc_DIR+"cycling_data_cell_"+cell_no+".csv")
    N1 = np.unique(data["Cycle number"])
    Ia = data["Current [mA]"].to_numpy()/1000
    ta = data["Time [s]"].to_numpy()
    AhTh = integrate.cumtrapz(abs(Ia), ta)/3600
    AhTh = np.append(AhTh,AhTh[-1])
    data["AhTh"] = AhTh
    Q_app = []
    max_exp = []
    min_exp = []
    Rs = []
    AhT = []
    for n in tqdm(N1):
        df = data[data["Cycle number"]==n]
        qmax = round(max(df["Capacity [Ah]"]),2)
        Q_app.append(qmax)
        emax = round(max(df["Expansion [mu m]"].fillna(0)),1)
        max_exp.append(emax)
        emin = round(min(df["Expansion [mu m]"].fillna(0)),1)
        min_exp.append(emin)
        AhT.append(round(df["AhTh"].iloc[0],1))
        t = df["Time [s]"].to_numpy()
        V = df["Voltage [V]"].to_numpy()
        I = df["Current [mA]"].to_numpy()/1000
        try:
            idxi1 = np.where((np.diff(I)<-1) & (I[:-1]>0))[0]
            idx = idxi1[0]
            R = (V[idx+1] - V[idx])/(I[idx+1] - I[idx])
            Rs.append(round(R,5))
        except:
            Rs.append(np.NaN)
    N = [n+1 for n in N1]
    dfs = pd.DataFrame({"N":N,"AhTh":AhT,"App Cap [Ah]":Q_app,"Min Exp [mu m]":min_exp,"Max Exp [mu m]":max_exp,"Rs [ohm]":Rs})
    mexp = round(dfs["Min Exp [mu m]"].iloc[0],1)
    dfs["Min Exp [mu m]"] = dfs["Min Exp [mu m]"] - mexp
    dfs["Min Exp [mu m]"] = dfs["Min Exp [mu m]"].round(1)
    dfs["Max Exp [mu m]"] = dfs["Max Exp [mu m]"] - mexp
    dfs["Max Exp [mu m]"] = dfs["Max Exp [mu m]"].round(1)
    dfs.to_csv(eSOH_DIR + "aging_param_cell_cyc_"+cell_no+".csv")

In [ ]:
fig,ax = plt.subplots(1,1,figsize=(5,4))
ax.plot(N,Q_app,'o')
ax.set_xlabel("Cycle Number")
ax.set_ylabel("Apparent Capacity [Ah]")
ax.set_title("Apparent Capacity 1 psi Cell")
plt.savefig(fig_DIR + "Apparent Cap.png")

In [ ]:
fig,ax = plt.subplots(1,1,figsize=(5,4))
ax.plot(N,min_exp,'o')
ax.plot(N,max_exp,'o')
ax.set_xlabel("Cycle Number")
ax.set_ylabel(r"Expansion [$\mu$m]")
ax.set_title("Expansion 1 psi Cell")
ax.legend(["Min","Max"])
plt.savefig(fig_DIR + "Expansion.png")

In [ ]:
fig,ax = plt.subplots(1,1,figsize=(5,4))
# ax.plot(data["Time [s]"],data["Capacity [Ah]"])
ax.plot(data["Time [s]"],data["Q [Ah]"])

In [ ]:

df= df.reset_index(drop=True)
fig,ax = plt.subplots(2,1,figsize=(4,6))
ax1 = ax.flat[0]
ax1.plot(t,V)
ax1.plot(t[idx],V[idx],"rx")
ax1.plot(t[idx+1],V[idx+1],"rx")
ax1.set_ylabel("Voltage [V]")
ax1.set_xlabel("Time [s]")
ax2 = ax.flat[1]
ax2.plot(t,I)
ax2.plot(t[idx],I[idx],"rx")
ax2.plot(t[idx+1],I[idx+1],"rx")
ax2.set_ylabel("Current [A]")
ax2.set_xlabel("Time [s]")
fig.suptitle(f"Cell: {cell_no}, Cycle #{n}")
fig.tight_layout()

In [ ]:
n = N1[0]
fig = plot_fun(data,n,cell_no)
plt.savefig(fig_DIR + "cycle 0.png")

## Plot Aging Metrics

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(5,4))
for cell in range(26,36):
    cell_no = f'{cell:02d}'
    df1 = pd.read_csv(eSOH_DIR + "aging_param_cell_" + cell_no + ".csv")
    ax.plot(df1["N"],df1["App Cap [Ah]"])
ax.legend(["1 PSI","5 PSI","5 PSI","10 PSI","10 PSI","15 PSI","15 PSI","20 PSI","20 PSI","25 PSI"])
ax.set_xlabel("Cycle Number")
ax.set_ylabel("Apparent Capacity [Ah]")
ax.set_title("Apparent Capacity")
plt.savefig(fig_DIR + "Apparent Cap All.png")

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(5,4))
for cell in range(26,36):
    cell_no = f'{cell:02d}'
    df1 = pd.read_csv(eSOH_DIR + "aging_param_cell_" + cell_no + ".csv")
    ax.plot(df1["N"],df1["Min Exp [mu m]"])
ax.legend(["1 PSI","5 PSI","5 PSI","10 PSI","10 PSI","15 PSI","15 PSI","20 PSI","20 PSI","25 PSI"])
ax.set_xlabel("Cycle Number")
ax.set_ylabel(r"Irreversible Expansion [$\mu$m]")
ax.set_title("Irreversible Expansion")
plt.savefig(fig_DIR + "Expansion All.png")

In [ ]:
df1.columns

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(5,4))
for cell in range(26,36):
    cell_no = f'{cell:02d}'
    df1 = pd.read_csv(eSOH_DIR + "aging_param_cell_" + cell_no + ".csv")
    ax.plot(df1["N"],df1["Rs [ohm]"])
ax.legend(["1 PSI","5 PSI","5 PSI","10 PSI","10 PSI","15 PSI","15 PSI","20 PSI","20 PSI","25 PSI"])
ax.set_xlabel("Cycle Number")
ax.set_ylabel(r"Resistance [$\Omega$]")
ax.set_title("Resistance")
plt.savefig(fig_DIR + "Resistance All.png")

In [ ]:
cells = [1,4,7,10]

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(5,4))
for cell in cells:
    cell_no = f'{cell:02d}'
    df1 = pd.read_csv(eSOH_DIR + "aging_param_cell_cyc_" + cell_no + ".csv")
    ax.plot(df1["N"],df1["App Cap [Ah]"])
ax.legend(["C/5","1.5C","2C","Mixed Crate"])
ax.set_xlabel("Cycle Number")
ax.set_ylabel("Apparent Capacity [Ah]")
ax.set_title("Apparent Capacity")
ax.set_ylim(1,5)
# plt.savefig(fig_DIR + "Apparent Cap All.png")

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(5,4))
for cell in cells:
    cell_no = f'{cell:02d}'
    df1 = pd.read_csv(eSOH_DIR + "aging_param_cell_cyc_" + cell_no + ".csv")
    ax.plot(df1["N"],df1["Min Exp [mu m]"])
ax.legend(["C/5","1.5C","2C","Mixed Crate"])
ax.set_xlabel("Cycle Number")
ax.set_ylabel(r"Irreversible Expansion [$\mu$m]")
ax.set_title("Irreversible Expansion")
plt.savefig(fig_DIR + "Expansion All.png")

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(5,4))
for cell in cells:
    cell_no = f'{cell:02d}'
    df1 = pd.read_csv(eSOH_DIR + "aging_param_cell_cyc_" + cell_no + ".csv")
    ax.plot(df1["N"],df1["Rs [ohm]"])
ax.legend(["C/5","1.5C","2C","Mixed Crate"])
ax.set_xlabel("Cycle Number")
ax.set_ylabel(r"Resistance [$\Omega$]")
ax.set_title("Resistance")
plt.savefig(fig_DIR + "Resistance All.png")

In [ ]:
for cell in [26]:
    cell_no = f'{cell:02d}'
    data = pd.read_csv(cyc_DIR+"cycling_data_cell_"+cell_no+".csv")
    N1 = np.unique(data["Cycle number"])
    Ia = data["Current [mA]"].to_numpy()/1000
    ta = data["Time [s]"].to_numpy()
    AhTh = integrate.cumtrapz(abs(Ia), ta)/3600
    AhTh = np.append(AhTh,AhTh[-1])
    data["AhTh"] = AhTh
    Q_app = []
    max_exp1 = []
    min_exp1 = []
    Rs = []
    AhT = []
    for n in tqdm(N1):
        df = data[data["Cycle number"]==n]
        qmax = round(max(df["Capacity [Ah]"]),2)
        Q_app.append(qmax)
        emax = round(max(df["Expansion [mu m]"].fillna(0)),1)
        max_exp1.append(emax)
        emin = round(min(df["Expansion [mu m]"].fillna(0)),1)
        min_exp1.append(emin)
        AhT.append(round(df["AhTh"].iloc[0],1))
        t = df["Time [s]"].to_numpy()
        V = df["Voltage [V]"].to_numpy()
        I = df["Current [mA]"].to_numpy()/1000
        try:
            idxi1 = np.where((np.diff(I)<-1) & (I[:-1]>0))[0]
            idx = idxi1[0]
            R = (V[idx+1] - V[idx])/(I[idx+1] - I[idx])
            Rs.append(round(R,5))
        except:
            Rs.append(np.NaN)
    N_1 = [n+1 for n in N1]
max_exp1 = np.array(max_exp1)
min_exp1 = np.array(min_exp1)
min_exp1_0 = min_exp1[0]
min_exp1 = min_exp1 - min_exp1[0]
max_exp1 = max_exp1 - min_exp1[0]

In [ ]:
for cell in [35]:
    cell_no = f'{cell:02d}'
    data = pd.read_csv(cyc_DIR+"cycling_data_cell_"+cell_no+".csv")
    N1 = np.unique(data["Cycle number"])
    Ia = data["Current [mA]"].to_numpy()/1000
    ta = data["Time [s]"].to_numpy()
    AhTh = integrate.cumtrapz(abs(Ia), ta)/3600
    AhTh = np.append(AhTh,AhTh[-1])
    data["AhTh"] = AhTh
    Q_app = []
    max_exp2 = []
    min_exp2 = []
    Rs = []
    AhT = []
    for n in tqdm(N1):
        df = data[data["Cycle number"]==n]
        qmax = round(max(df["Capacity [Ah]"]),2)
        Q_app.append(qmax)
        emax = round(max(df["Expansion [mu m]"].fillna(0)),1)
        max_exp2.append(emax)
        emin = round(min(df["Expansion [mu m]"].fillna(0)),1)
        min_exp2.append(emin)
        AhT.append(round(df["AhTh"].iloc[0],1))
    N_2 = [n+1 for n in N1]
max_exp2 = np.array(max_exp2)
min_exp2 = np.array(min_exp2)
min_exp2_0 = min_exp2[0]
min_exp2 = min_exp2 - min_exp2_0
max_exp2 = max_exp2 - min_exp2_0

In [ ]:
fig,ax = plt.subplots(1,1,figsize=(5,4))
ax.plot(N_1,min_exp1,'o',markevery=5,color="hotpink")
ax.plot(N_1,max_exp1,'x',markevery=5,color="hotpink")
ax.plot(N_2,min_exp2,'o',markevery=5,color="black")
ax.plot(N_2,max_exp2,'x',markevery=5,color="black")
ax.set_xlabel("Cycle Number")
ax.set_ylabel(r"Expansion [$\mu$m]")
ax.set_title("Minimum and Maximum Expansion")
ax.legend(["0 kPa 0% SOC","0 kPa 100% SOC","172 kPa 0% SOC","172 kPa 100% SOC"])
plt.savefig(fig_DIR + "expansion_min_max.png")

In [ ]:
fig,ax = plt.subplots(1,1,figsize=(5,4))
ax.plot(N_1,min_exp1,'o',markevery=5,color="#ffc6c4")
ax.plot(N_1,max_exp1,'x',markevery=5,color="#ffc6c4")
ax.plot(N_2,min_exp2,'o',markevery=5,color="#f1807e")
ax.plot(N_2,max_exp2,'x',markevery=5,color="#f1807e")
ax.set_xlabel("Cycle Number")
ax.set_ylabel(r"Expansion [$\mu$m]")
ax.set_title("Minimum and Maximum Expansion")
ax.legend(["0 psi Min","0 psi Max","5 psi Min","5 psi Max"])
plt.savefig(fig_DIR + "expansion_min_max.png")

In [ ]:
colors = ["violet","salmon","darkorange","red","darkred","black"]
fig,ax = plt.subplots(1,1)
i = 0
for cell in [26,9,29,31,33,35]:
    cell_no = f'{cell:02d}'
    data = pd.read_csv(cyc_DIR+"cycling_data_cell_"+cell_no+".csv")
    N1 = np.unique(data["Cycle number"])
    Ia = data["Current [mA]"].to_numpy()/1000
    ta = data["Time [s]"].to_numpy()
    AhTh = integrate.cumtrapz(abs(Ia), ta)/3600
    AhTh = np.append(AhTh,AhTh[-1])
    data["AhTh"] = AhTh
    Q_app = []
    max_exp = []
    min_exp = []
    Rs = []
    AhT = []
    for n in tqdm(N1):
        df = data[data["Cycle number"]==n]
        qmax = round(max(df["Capacity [Ah]"]),2)
        Q_app.append(qmax)
        emax = round(max(df["Expansion [mu m]"].fillna(0)),1)
        max_exp.append(emax)
        emin = round(min(df["Expansion [mu m]"].fillna(0)),1)
        min_exp.append(emin)
        AhT.append(round(df["AhTh"].iloc[0],1))
        t = df["Time [s]"].to_numpy()
        V = df["Voltage [V]"].to_numpy()
        I = df["Current [mA]"].to_numpy()/1000
        try:
            idxi1 = np.where((np.diff(I)<-1) & (I[:-1]>0))[0]
            idx = idxi1[0]
            R = (V[idx+1] - V[idx])/(I[idx+1] - I[idx])
            Rs.append(round(R,5))
        except:
            Rs.append(np.NaN)
    N = [n+1 for n in N1]
    dfs = pd.DataFrame({"N":N,"AhTh":AhT,"App Cap [Ah]":Q_app,"Min Exp [mu m]":min_exp,"Max Exp [mu m]":max_exp,"Rs [ohm]":Rs})
    mexp = round(dfs["Min Exp [mu m]"].iloc[0],1)
    dfs["Min Exp [mu m]"] = dfs["Min Exp [mu m]"] - mexp
    dfs["Min Exp [mu m]"] = dfs["Min Exp [mu m]"].round(1)
    dfs["Max Exp [mu m]"] = dfs["Max Exp [mu m]"] - mexp
    dfs["Max Exp [mu m]"] = dfs["Max Exp [mu m]"].round(1)
    # dfs.to_csv(eSOH_DIR + "aging_param_cell_cyc_"+cell_no+".csv")
    ax.plot(dfs["N"],dfs["App Cap [Ah]"],color=colors[i])
    i+=1
ax.legend(['0 kPa','35 kPa','69 kPa','103 kPa','138 kPa','172 kPa'])
ax.set_xlabel("Cycle Number")
ax.set_ylabel("Apparent Capacity [Ah]")

In [ ]:
cell = 33

cell_no = f'{cell:02d}'
data = pd.read_csv(cyc_DIR+"cycling_data_cell_"+cell_no+".csv")
N1 = np.unique(data["Cycle number"])
Ia = data["Current [mA]"].to_numpy()/1000
ta = data["Time [s]"].to_numpy()
Ea = data["Expansion [mu m]"].to_numpy()
Va = data["Voltage [V]"].to_numpy()

In [ ]:
N1

In [ ]:
fig,ax = plt.subplots(3,1, figsize=(15,12))
ax1 = ax.flat[0]
ax1.plot(ta,Ia)
ax1.set_title("Current")
ax2 = ax.flat[1]
ax2.plot(ta,Va)
ax1.set_title("Voltage")
ax3 = ax.flat[2]
ax3.plot(ta,Ea)
ax1.set_title("Expansion")

## Plot Repeats

In [ ]:
cell = 26
dfe_0=pd.read_csv(eSOH_DIR+f"aging_param_cell_cyc_{cell:02d}.csv")
cell = 27
dfe_11=pd.read_csv(eSOH_DIR+f"aging_param_cell_cyc_{cell:02d}.csv")
cell = 28
dfe_12=pd.read_csv(eSOH_DIR+f"aging_param_cell_cyc_{cell:02d}.csv")
cell = 9
dfe_13=pd.read_csv(eSOH_DIR+f"aging_param_cell_cyc_{cell:02d}.csv")
cell = 29
dfe_21=pd.read_csv(eSOH_DIR+f"aging_param_cell_cyc_{cell:02d}.csv")
cell = 30
dfe_22=pd.read_csv(eSOH_DIR+f"aging_param_cell_cyc_{cell:02d}.csv")
cell = 31
dfe_31=pd.read_csv(eSOH_DIR+f"aging_param_cell_cyc_{cell:02d}.csv")
cell = 32
dfe_32=pd.read_csv(eSOH_DIR+f"aging_param_cell_cyc_{cell:02d}.csv")
cell = 33
dfe_41=pd.read_csv(eSOH_DIR+f"aging_param_cell_cyc_{cell:02d}.csv")
cell = 34
dfe_42=pd.read_csv(eSOH_DIR+f"aging_param_cell_cyc_{cell:02d}.csv")
cell = 35
dfe_5=pd.read_csv(eSOH_DIR+f"aging_param_cell_cyc_{cell:02d}.csv")

In [ ]:
fig, ax = plt.subplots(1,3,figsize=(15,4))
ax1 = ax.flat[0]
ax1.plot(dfe_11["N"],dfe_11["App Cap [Ah]"])
ax1.plot(dfe_12["N"],dfe_12["App Cap [Ah]"])
ax1.plot(dfe_13["N"],dfe_13["App Cap [Ah]"])
ax1.legend(["1","2","3"])
ax1.set_xlabel("Cycle Number")
ax1.set_ylabel("Capacity [Ah]")
ax2 = ax.flat[1]
ax2.plot(dfe_11["N"],dfe_11["Rs [ohm]"])
ax2.plot(dfe_12["N"],dfe_12["Rs [ohm]"])
ax2.plot(dfe_13["N"],dfe_13["Rs [ohm]"])
ax2.legend(["1","2","3"])
ax2.set_xlabel("Cycle Number")
ax2.set_ylabel(r"Resistance [$\Omega$]")
ax3 = ax.flat[2]
ax3.plot(dfe_11["N"],dfe_11["Min Exp [mu m]"])
ax3.plot(dfe_12["N"],dfe_12["Min Exp [mu m]"])
ax3.plot(dfe_13["N"],dfe_13["Min Exp [mu m]"])
ax3.legend(["1","2","3"])
ax3.set_xlabel("Cycle Number")
ax3.set_ylabel(r"Expansion [$\mu$m]")
fig.suptitle("5 psi")
plt.savefig(fig_DIR + "repeats_5psi.png")

In [ ]:
fig, ax = plt.subplots(1,3,figsize=(15,4))
ax1 = ax.flat[0]
ax1.plot(dfe_21["N"],dfe_21["App Cap [Ah]"])
ax1.plot(dfe_22["N"],dfe_22["App Cap [Ah]"])
ax1.legend(["1","2","3"])
ax1.set_xlabel("Cycle Number")
ax1.set_ylabel("Capacity [Ah]")
ax2 = ax.flat[1]
ax2.plot(dfe_21["N"],dfe_21["Rs [ohm]"])
ax2.plot(dfe_22["N"],dfe_22["Rs [ohm]"])
ax2.legend(["1","2","3"])
ax2.set_xlabel("Cycle Number")
ax2.set_ylabel(r"Resistance [$\Omega$]")
ax3 = ax.flat[2]
ax3.plot(dfe_21["N"],dfe_21["Min Exp [mu m]"])
ax3.plot(dfe_22["N"],dfe_22["Min Exp [mu m]"])
ax3.legend(["1","2","3"])
ax3.set_xlabel("Cycle Number")
ax3.set_ylabel(r"Expansion [$\mu$m]")
fig.suptitle("10 psi")
plt.savefig(fig_DIR + "repeats_10psi.png")

In [ ]:
fig, ax = plt.subplots(1,3,figsize=(15,4))
ax1 = ax.flat[0]
ax1.plot(dfe_31["N"],dfe_31["App Cap [Ah]"])
ax1.plot(dfe_32["N"],dfe_32["App Cap [Ah]"])
ax1.legend(["1","2","3"])
ax1.set_xlabel("Cycle Number")
ax1.set_ylabel("Capacity [Ah]")
ax2 = ax.flat[1]
ax2.plot(dfe_31["N"],dfe_31["Rs [ohm]"])
ax2.plot(dfe_32["N"],dfe_32["Rs [ohm]"])
ax2.legend(["1","2","3"])
ax2.set_xlabel("Cycle Number")
ax2.set_ylabel(r"Resistance [$\Omega$]")
ax3 = ax.flat[2]
ax3.plot(dfe_31["N"],dfe_31["Min Exp [mu m]"])
ax3.plot(dfe_32["N"],dfe_32["Min Exp [mu m]"])
ax3.legend(["1","2","3"])
ax3.set_xlabel("Cycle Number")
ax3.set_ylabel(r"Expansion [$\mu$m]")
fig.suptitle("15 psi")
plt.savefig(fig_DIR + "repeats_15psi.png")

In [ ]:
fig, ax = plt.subplots(1,3,figsize=(15,4))
ax1 = ax.flat[0]
ax1.plot(dfe_41["N"],dfe_41["App Cap [Ah]"])
ax1.plot(dfe_42["N"],dfe_42["App Cap [Ah]"])
ax1.legend(["1","2","3"])
ax1.set_xlabel("Cycle Number")
ax1.set_ylabel("Capacity [Ah]")
ax2 = ax.flat[1]
ax2.plot(dfe_41["N"],dfe_41["Rs [ohm]"])
ax2.plot(dfe_42["N"],dfe_42["Rs [ohm]"])
ax2.legend(["1","2","3"])
ax2.set_xlabel("Cycle Number")
ax2.set_ylabel(r"Resistance [$\Omega$]")
ax3 = ax.flat[2]
ax3.plot(dfe_41["N"],dfe_41["Min Exp [mu m]"])
ax3.plot(dfe_42["N"],dfe_42["Min Exp [mu m]"])
ax3.legend(["1","2","3"])
ax3.set_xlabel("Cycle Number")
ax3.set_ylabel(r"Expansion [$\mu$m]")
fig.suptitle("20 psi")
plt.savefig(fig_DIR + "repeats_20psi.png")